In [1]:
import os
os.environ["HF_HUB_READ_TIMEOUT"] = "60"
os.environ["HF_HUB_CONNECT_TIMEOUT"] = "60"
from datasets import load_dataset

In [2]:
train_dataset = load_dataset('slegroux/tiny-imagenet-200-clean', split='train')                
valid_dataset = load_dataset('slegroux/tiny-imagenet-200-clean', split='validation')
test_dataset = load_dataset('slegroux/tiny-imagenet-200-clean', split='test')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/151M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/7.54M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/7.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/98179 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4909 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4923 [00:00<?, ? examples/s]

In [3]:
import torch
import torch.nn as nn
from collections import OrderedDict
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
device

device(type='cuda')

In [6]:
class MobileNet_v1(nn.Module):
    def __init__(self, num_classes):
        super(MobileNet_v1, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=(3,3), stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU(inplace=True)
        
        self.conv1_dw = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(3,3), stride=1, groups=32, padding=1, bias=False)
        self.bn1_dw = nn.BatchNorm2d(32)
        self.relu1_dw = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1,1), stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU(inplace=True)
        
        self.conv2_dw = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3,3), groups=64, stride=2, padding = 1,bias=False)
        self.bn2_dw = nn.BatchNorm2d(64)
        self.relu2_dw = nn.ReLU(inplace=True)
        
        self.conv3 = nn.Conv2d(in_channels=64,out_channels=128, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU(inplace=True)
        
        self.conv3_dw = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=(3,3), stride=1, padding=1, groups=128,bias=False)
        self.bn3_dw = nn.BatchNorm2d(128)
        self.relu3_dw = nn.ReLU(inplace=True)
        
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn4 = nn.BatchNorm2d(128)
        self.relu4 = nn.ReLU(inplace=True)
        
        self.conv4_dw = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=(3,3), stride=2, padding=1, groups=128,bias=False)
        self.bn4_dw = nn.BatchNorm2d(128)
        self.relu4_dw = nn.ReLU(inplace=True)
        
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn5 = nn.BatchNorm2d(256)
        self.relu5 = nn.ReLU(inplace=True)
        
        self.conv5_dw = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(3,3), stride=1, padding=1, groups=256,bias=False)
        self.bn5_dw = nn.BatchNorm2d(256)
        self.relu5_dw = nn.ReLU(inplace=True)
        
        self.conv6 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn6 = nn.BatchNorm2d(256)
        self.relu6 = nn.ReLU(inplace=True)
        
        self.conv6_dw = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(3,3), stride=2, padding=1, groups=256,bias=False)
        self.bn6_dw = nn.BatchNorm2d(256)
        self.relu6_dw = nn.ReLU(inplace=True)
        
        self.conv7 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn7 = nn.BatchNorm2d(512)
        self.relu7 = nn.ReLU(inplace=True)
        
        self.conv8_dw_1 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), stride=1, padding=1, groups=512,bias=False)
        self.bn8_dw_1 = nn.BatchNorm2d(512)
        self.relu8_dw_1 = nn.ReLU(inplace=True)
        
        self.conv9_1 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn9_1 = nn.BatchNorm2d(512)
        self.relu9_1 = nn.ReLU(inplace=True)
        
        self.conv8_dw_2 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), stride=1, padding=1, groups=512,bias=False)
        self.bn8_dw_2 = nn.BatchNorm2d(512)
        self.relu8_dw_2 = nn.ReLU(inplace=True)
        
        self.conv9_2 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn9_2 = nn.BatchNorm2d(512)
        self.relu9_2 = nn.ReLU(inplace=True)
        
        self.conv8_dw_3 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), stride=1, padding=1, groups=512,bias=False)
        self.bn8_dw_3 = nn.BatchNorm2d(512)
        self.relu8_dw_3 = nn.ReLU(inplace=True)
        
        self.conv9_3 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn9_3 = nn.BatchNorm2d(512)
        self.relu9_3 = nn.ReLU(inplace=True)
        
        self.conv8_dw_4 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), stride=1, padding=1, groups=512,bias=False)
        self.bn8_dw_4 = nn.BatchNorm2d(512)
        self.relu8_dw_4 = nn.ReLU(inplace=True)
        
        self.conv9_4 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn9_4 = nn.BatchNorm2d(512)
        self.relu9_4 = nn.ReLU(inplace=True)
        
        self.conv8_dw_5 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), stride=1, padding=1, groups=512,bias=False)
        self.bn8_dw_5 = nn.BatchNorm2d(512)
        self.relu8_dw_5 = nn.ReLU(inplace=True)
        
        self.conv9_5 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn9_5 = nn.BatchNorm2d(512)
        self.relu9_5 = nn.ReLU(inplace=True)
        
        self.conv9_dw = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), stride=2, padding=1, groups=512,bias=False)
        self.bn9 = nn.BatchNorm2d(512)
        self.relu9_dw = nn.ReLU(inplace=True)
        
        self.conv10 = nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn10 = nn.BatchNorm2d(1024)
        self.relu10 = nn.ReLU(inplace=True)
        
        self.conv10_dw = nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=(3,3), stride=1, padding=1, groups=1024,bias=False)
        self.bn10_dw = nn.BatchNorm2d(1024)
        self.relu10_dw = nn.ReLU(inplace=True)
        
        self.conv11 = nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=(1,1), stride=1, padding=0,bias=False)
        self.bn11 = nn.BatchNorm2d(1024)
        self.relu11 = nn.ReLU(inplace=True)
        
        self.avgpool = nn.AvgPool2d(kernel_size=(7,7), stride=1)
        self.linear = nn.Linear(1024,num_classes)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu1_dw(self.bn1_dw(self.conv1_dw(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.relu2_dw(self.bn2_dw(self.conv2_dw(x)))
        x = self.relu3(self.bn3(self.conv3(x)))
        x = self.relu3_dw(self.bn3_dw(self.conv3_dw(x)))
        x = self.relu4(self.bn4(self.conv4(x)))
        x = self.relu4_dw(self.bn4_dw(self.conv4_dw(x)))
        x = self.relu5(self.bn5(self.conv5(x)))
        x = self.relu5_dw(self.bn5_dw(self.conv5_dw(x)))
        x = self.relu6(self.bn6(self.conv6(x)))
        x = self.relu6_dw(self.bn6_dw(self.conv6_dw(x)))
        x = self.relu7(self.bn7(self.conv7(x)))
    
        # 5× blocks
        for i in range(1, 6):
            x = getattr(self, f"relu8_dw_{i}")(getattr(self, f"bn8_dw_{i}")(getattr(self, f"conv8_dw_{i}")(x)))
            x = getattr(self, f"relu9_{i}")(getattr(self, f"bn9_{i}")(getattr(self, f"conv9_{i}")(x)))
    
        x = self.relu9_dw(self.bn9(self.conv9_dw(x)))
        x = self.relu10(self.bn10(self.conv10(x)))
        x = self.relu10_dw(self.bn10_dw(self.conv10_dw(x)))
        x = self.relu11(self.bn11(self.conv11(x)))
    
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.linear(x)
        return x


In [7]:
model = MobileNet_v1(200).to(device)

In [8]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def preprocess(example, transform):
    example["image"] = [transform(img) for img in example["image"]]
    return example

train_dataset = train_dataset.with_transform(lambda x: preprocess(x, transform))
valid_dataset = valid_dataset.with_transform(lambda x: preprocess(x, test_transform))
test_dataset = test_dataset.with_transform(lambda x: preprocess(x, test_transform))

In [9]:
optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.1,
            momentum=0.9,
            weight_decay=1e-4,
            nesterov=True
        )

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader   = DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=4)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[30, 60, 120],
    gamma=0.1
)
criterion = nn.CrossEntropyLoss()

In [10]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:
        images = batch['image'].to(device)
        label = batch['label'].to(device)

        # Forward
        output = model(images)
        loss = criterion(output, label)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, preds = output.max(1)
        correct += preds.eq(label).sum().item()
        total += label.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch in dataloader:
        images = batch['image'].to(device)
        label = batch['label'].to(device)

        # Forward
        output = model(images)
        loss = criterion(output, label)
        running_loss += loss.item() * images.size(0)
        _, preds = output.max(1)
        correct += preds.eq(label).sum().item()
        total += label.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [11]:
best_val_acc = 0.0
num_epochs = 50
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc = validate(
        model, val_loader, criterion, device
    )
    
    scheduler.step() 
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )


Epoch [1/50] Train Loss: 4.4695, Train Acc: 0.0723 | Val Loss: 4.5078, Val Acc: 0.0966
Epoch [2/50] Train Loss: 3.5415, Train Acc: 0.1969 | Val Loss: 3.4207, Val Acc: 0.2088
Epoch [3/50] Train Loss: 3.0241, Train Acc: 0.2921 | Val Loss: 3.0685, Val Acc: 0.3037
Epoch [4/50] Train Loss: 2.7000, Train Acc: 0.3562 | Val Loss: 2.8318, Val Acc: 0.3312
Epoch [5/50] Train Loss: 2.4801, Train Acc: 0.4010 | Val Loss: 2.6640, Val Acc: 0.3665
Epoch [6/50] Train Loss: 2.3294, Train Acc: 0.4336 | Val Loss: 2.6178, Val Acc: 0.3769
Epoch [7/50] Train Loss: 2.2077, Train Acc: 0.4580 | Val Loss: 2.4580, Val Acc: 0.4168
Epoch [8/50] Train Loss: 2.1149, Train Acc: 0.4774 | Val Loss: 2.3374, Val Acc: 0.4400
Epoch [9/50] Train Loss: 2.0451, Train Acc: 0.4917 | Val Loss: 2.5938, Val Acc: 0.4042
Epoch [10/50] Train Loss: 1.9887, Train Acc: 0.5044 | Val Loss: 2.3534, Val Acc: 0.4343
Epoch [11/50] Train Loss: 1.9376, Train Acc: 0.5141 | Val Loss: 2.2919, Val Acc: 0.4500
Epoch [12/50] Train Loss: 1.8996, Train A

In [12]:
model_best = MobileNet_v1(200)
state_dict = torch.load("best_model.pth", map_location="cuda")
model.load_state_dict(state_dict)
# _, test_acc = validate(model, test_loader, criterion, device)
# print(test_acc)

<All keys matched successfully>

In [13]:
_, test_acc = validate(model, test_loader, criterion, device)
print(test_acc)

0.6146658541539711
